In [4]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
import numpy as np

In [5]:
import numpy as np
ratings = np.array([
    [1,1,0,1,0,0],
    [1,1,0,1,1,0],
    [0,0,1,0,1,1],
    [1,0,0,1,0,1],
    [0,1,1,0,1,1],
    [1,1,0,0,0,1]
], dtype='float32')

print(ratings)

[[1. 1. 0. 1. 0. 0.]
 [1. 1. 0. 1. 1. 0.]
 [0. 0. 1. 0. 1. 1.]
 [1. 0. 0. 1. 0. 1.]
 [0. 1. 1. 0. 1. 1.]
 [1. 1. 0. 0. 0. 1.]]


In [6]:
x_train = ratings
num_movies = x_train.shape[1]
latent_dim = 2

In [7]:
inputs = layers.Input(shape=(num_movies,))
h = layers.Dense(16,activation='relu')(inputs)

z_mean = layers.Dense(latent_dim)(h)
z_log_var = layers.Dense(latent_dim)(h)

In [ ]:
# class Sampling(layers.Layer):
#     def call(self, inputs):
#         z_mean, z_log_var = inputs
#         epsilon = tf.random.normal(
#             shape=tf.shape(z_mean)
#         )
#         return (
#             z_mean +
#             tf.exp(0.5 * z_log_var)
#             * epsilon
#         )

In [9]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    def compute_output_shape(self, input_shape):
        return input_shape[0]


z = Sampling()([z_mean, z_log_var])

In [12]:
z = Sampling()([z_mean,z_log_var])

In [13]:
encoder = Model(inputs,[z_mean, z_log_var, z])

In [14]:
#Dcoder

latent_inputs = layers.Input(shape=(latent_dim,))

x = layers.Dense(16,activation='relu')(latent_inputs)

outputs = layers.Dense( num_movies,activation='sigmoid')(x)

decoder = Model(latent_inputs,outputs)

In [15]:
class VAE(Model):
    def __init__(
        self,
        encoder,
        decoder
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)

            reconstruction = self.decoder(z)

            reconstruction_loss = (
                tf.reduce_mean(
                    tf.keras.losses.binary_crossentropy(
                        data,
                        reconstruction
                    )
                ) * num_movies
            )

            kl_loss = (
                -0.5 *
                tf.reduce_mean(
                    tf.reduce_sum(
                        1
                        + z_log_var
                        - tf.square(z_mean)
                        - tf.exp(z_log_var),
                        axis=1
                    )
                )
            )

            total_loss = (
                reconstruction_loss +
                kl_loss
            )

        grads = tape.gradient(
            total_loss,
            self.trainable_variables
        )

        self.optimizer.apply_gradients(
            zip(
                grads,
                self.trainable_variables
            )
        )

        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss
        }


In [16]:
vae = VAE(
    encoder,
    decoder
)
vae.compile(
    optimizer='adam'
)
vae.fit(
    x_train,
    epochs=200,
    verbose=1
)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 906ms/step - kl_loss: 0.1500 - loss: 4.3877 - reconstruction_loss: 4.2377
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - kl_loss: 0.1438 - loss: 4.1876 - reconstruction_loss: 4.0438
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - kl_loss: 0.1379 - loss: 4.2116 - reconstruction_loss: 4.0737
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - kl_loss: 0.1323 - loss: 4.2728 - reconstruction_loss: 4.1405
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - kl_loss: 0.1268 - loss: 4.1109 - reconstruction_loss: 3.9840
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - kl_loss: 0.1218 - loss: 4.2611 - reconstruction_loss: 4.1393
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - kl_loss: 0.1170 - loss: 4.1925 - reconstruction_loss: 4.0755
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - kl_loss: 0.1124 - loss: 4.1181 - reconstruction_loss: 4.0057
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - kl_loss: 0.1078 - loss: 4.3722 - reconstruc

In [17]:
user = np.array([
    [1,1,0,1,0,0]
], dtype='float32')

In [18]:
z_mean, z_log_var, z = encoder.predict(user)
recommendation = decoder.predict(z)
print(recommendation)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
[[0.562312   0.76874185 0.28755105 0.47519997 0.44389126 0.6438715 ]]
